In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q faiss-cpu sentence-transformers PyPDF2 pdf2image pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (231 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 7.pdf to 7 (1).pdf


In [ ]:
import re

# Remove code-like noise
clean_text = re.sub(r'[^a-zA-Z0-9.,?\n ]', ' ', text)

# Remove extra spaces
clean_text = re.sub(r'\s+', ' ', clean_text)

print("Clean text length:", len(clean_text))

Clean text length: 4106


In [ ]:
import re

# Clean text
clean_text = re.sub(r'[^a-zA-Z0-9.,?\n ]', ' ', text)
clean_text = re.sub(r'\s+', ' ', clean_text)

# Split into sentences
sentences = clean_text.split('.')

chunks = []

for s in sentences:
    s = s.strip()

    # ❌ REMOVE BAD LINES
    if len(s) < 40:
        continue
    if "print" in s or "import" in s or "code" in s.lower():
        continue
    if "output" in s.lower():
        continue

    chunks.append(s)

print("Total clean chunks:", len(chunks))
print("\nSample:", chunks[:3])

Total clean chunks: 18

Sample: ['EXP 7 Integrate a Vector Database with an LLM to build Retrieval Augmented Generation RAG system that answers questions based on external PDF documents', 'Dataset LinkedIn Profile or Research Paper PDFs', 'from pretrained enc name enc model AutoModel']


In [ ]:
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
import re

pdf_path = list(uploaded.keys())[0]

text = ""

# ===== TRY NORMAL PDF EXTRACTION =====
reader = PdfReader(pdf_path)
for page in reader.pages:
    if page.extract_text():
        text += page.extract_text()

# ===== IF FAILED → USE OCR =====
if len(text.strip()) < 50:
    print("Using OCR...")
    images = convert_from_path(pdf_path)
    for img in images:
        text += pytesseract.image_to_string(img)

print("Raw text length:", len(text))

# ===== CLEAN TEXT (REMOVE NOISE) =====
clean_text = re.sub(r'[^a-zA-Z0-9.,?\n ]', ' ', text)
clean_text = re.sub(r'\s+', ' ', clean_text)

print("Clean text length:", len(clean_text))

# ===== CHUNKING (GOOD QUALITY) =====
chunks = []

for i in range(0, len(clean_text), 800):
    chunk = clean_text[i:i+800]
    if len(chunk.strip()) > 200:
        chunks.append(chunk)

print("Total chunks:", len(chunks))

# Preview
print("\nSample chunk:\n", chunks[0][:300])

Using OCR...
Raw text length: 4762
Clean text length: 4106
Total chunks: 5

Sample chunk:
 EXP 7 Integrate a Vector Database with an LLM to build Retrieval Augmented Generation RAG system that answers questions based on external PDF documents. Dataset LinkedIn Profile or Research Paper PDFs. CODE Retrieval Augmented Generation RAG over PDF Documents LLM TinyLlama TinyLlama 1.1B Chat v1.0 


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (5, 384)


In [ ]:
import faiss

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS Ready!")

FAISS Ready!


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_name)

print("LLM Loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM Loaded!


In [ ]:
def ask_question(query, k=2):

    query_vec = model.encode([query])
    query_vec = np.array(query_vec)

    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, k)

    context = ""
    for idx in indices[0]:
        context += chunks[idx][:400] + "\n"

    print("\n--- Retrieved Context ---\n")
    print(context)

    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question: {query}
Answer:
"""

    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    output = llm_model.generate(**inputs, max_new_tokens=50)

    answer = llm_tokenizer.decode(output[0], skip_special_tokens=True)

    print("\n--- Final Answer ---\n")
    print(answer)


# 🔥 ASK HERE
ask_question("What is the aim of this experiment?")


--- Retrieved Context ---

ip special tokens True return output text.split Answer 1 .strip RUN QUERY question What is the main contribution of this paper? answer generate rag answer question print nQuestion , question print nAnswer , answer OUTPUT Total chunks 9927 This is a friendly reminder the current text generation call has exceeded the model s predefined maximum length 2048 . Depending on the model, you may observe ex
EXP 7 Integrate a Vector Database with an LLM to build Retrieval Augmented Generation RAG system that answers questions based on external PDF documents. Dataset LinkedIn Profile or Research Paper PDFs. CODE Retrieval Augmented Generation RAG over PDF Documents LLM TinyLlama TinyLlama 1.1B Chat v1.0 Google Colab Ready FULL CODE IN ONE BLOCK import faiss import numpy as np import torch from transfor


--- Final Answer ---


Answer the question using ONLY the context below.

Context:
ip special tokens True return output text.split Answer 1 .strip RUN QUERY question Wh